# Gradient Accumulation 

A technique that allows us to perform multiple forward and backward passes, but instead of updating the weights immediately, we **accumulate** the gradients over **several steps**. Only after the specified number of `accumulation steps` do we trigger the optimizer to update the weights. This allows us to simulate a massive Global Batch Size that would otherwise not fit in our hardware's VRAM.

- **Global Batch** (`curr_global_batch_tokens`): The total number of tokens the optimizer needs to look at together to calculate an accurate gradient. 
- **Micro Batch** (`curr_micro_batch_size`): The number of sequences your hardware can hold in its VRAM (GPU Memory) during a single forward and backward pass.
  - Example: if `curr_micro_batch_size` $=2$, and `seq_len` $=32$, then we are processing 64 tokens for that forward pass
  - Once we have accumulated enough gradients to reach `curr_global_batch_tokens`, we scale it, this makes it seem like we did a normal curr_global_batch_tokens pass. 

$$\text{Scaled Loss} = \frac{\text{Loss}}{\text{Accumulation Steps}}$$

In [1]:
import math
import torch

In [ ]:
class GradientAccumulation:
    """
    Handles gradient accumulation.
    """

    def __init__(self):
        self.accumulation_steps = 1
        self.current_micro_step = 0

    def update_shape_target(
        self, global_tokens: int, micro_batch_size: int, seq_len: int
    ):
        """Calculate how many micro-batches are needed to hit the global token target (batch-size)."""
        micro_capacity = micro_batch_size * seq_len
        self.accumulation_steps = math.ceil(global_tokens / micro_capacity)
        self.current_micro_step = 0  # Reset on shape change
        return self.accumulation_steps

    def scale_loss(self, loss: torch.Tensor) -> torch.Tensor:
        """Scale the loss so accumulated gradients average correctly."""
        return loss / self.accumulation_steps
    
    @property
    def should_step(self)->bool:
        """Returns True if the accumulation target (global_batch_tokens) is met."""
        return self.current_micro_step != 0 and self.current_micro_step % self.accumulation_steps == 0
